# 02 — Classical Baseline: Pixel-Mean SSIM Detector

**Author:** Sherif Abd El-Rady

Before building a deep learning solution, we establish a strong classical baseline. This forces honest evaluation: if the classical method achieves 0.85 AUC, the deep method must substantially exceed it to justify the added complexity.

## Method

1. Compute the **pixel-wise mean** and **standard deviation** of all normal training images.
2. For each test image, compute the **standardised pixel deviation**: `|img - mean| / std`.
3. Collapse channels → per-pixel anomaly score → image-level score (mean).
4. Evaluate with ROC-AUC and Average Precision.

## Limitations (and why PatchCore addresses them)

| Limitation | PatchCore fix |
|---|---|
| Only pixel intensity — no structural features | Deep CNN features capture texture + structure |
| Global mean collapses spatial variation | Per-patch embedding retains spatial context |
| Sensitive to image alignment | Patch k-NN is shift-tolerant |
| Requires uniform surfaces | Works on arbitrary textures and objects |


In [ ]:
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

SRC = Path('..') / 'src'
if str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

from defectdetect import ClassicalBaseline, AnomalyEvaluator
from defectdetect.scorer import AnomalyScorer

ROOT     = Path('..').resolve()
MVTEC    = ROOT / 'data' / 'mvtec'
CATEGORY = 'bottle'

# Auto-fall back to synthetic
if not (MVTEC / CATEGORY / 'train' / 'good').exists():
    MVTEC    = ROOT / 'data' / 'synthetic'
    CATEGORY = ''
    print('Using synthetic dataset')

base      = MVTEC / CATEGORY if CATEGORY else MVTEC
train_dir = base / 'train' / 'good'
test_dir  = base / 'test'
print('Ready')

In [ ]:
# ── 2.1  Load training images ─────────────────────────────────────────────────
train_paths = sorted(train_dir.glob('*.png')) + sorted(train_dir.glob('*.jpg'))
train_imgs  = [cv2.imread(str(p)) for p in train_paths]
print(f'Training images: {len(train_imgs)}')

In [ ]:
# ── 2.2  Fit the baseline on normal images ────────────────────────────────────
baseline = ClassicalBaseline(image_size=(256, 256), blur_sigma=2.0)
baseline.fit(train_imgs)

# Visualise the mean normal image
mean_vis = (baseline._mean_normal * 255).astype(np.uint8)
std_vis  = (baseline._std_normal  * 255 * 5).clip(0, 255).astype(np.uint8)  # amplified

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(cv2.cvtColor(train_imgs[0], cv2.COLOR_BGR2RGB))
axes[0].set_title('Sample training image')
axes[1].imshow(cv2.cvtColor(mean_vis, cv2.COLOR_BGR2RGB))
axes[1].set_title('Pixel-wise mean normal image')
axes[2].imshow(cv2.cvtColor(std_vis, cv2.COLOR_BGR2RGB))
axes[2].set_title('Pixel-wise std × 5 (bright = high variance region)')
for ax in axes: ax.axis('off')
plt.suptitle('Baseline fit: normal image statistics', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.3  Evaluate on the test set ─────────────────────────────────────────────
all_scores, all_labels, all_amaps = [], [], []

for cls_dir in sorted(test_dir.iterdir()):
    if not cls_dir.is_dir(): continue
    label = 0 if cls_dir.name == 'good' else 1
    paths = sorted(cls_dir.glob('*.png')) + sorted(cls_dir.glob('*.jpg'))
    for p in paths:
        img = cv2.imread(str(p))
        if img is None: continue
        score, amap = baseline.predict(img)
        all_scores.append(score)
        all_labels.append(label)
        all_amaps.append(amap)

scores = np.array(all_scores)
labels = np.array(all_labels)
print(f'Evaluated: {len(scores)} images ({labels.sum()} defect / {(labels==0).sum()} normal)')

In [ ]:
# ── 2.4  Image-level metrics ──────────────────────────────────────────────────
evaluator = AnomalyEvaluator()
evaluator.fit(scores, labels)
metrics = evaluator.image_level_metrics()

print('=== Classical Baseline — Image-Level ===')
print(f'ROC-AUC           : {metrics["roc_auc"]:.4f}')
print(f'Average Precision : {metrics["average_precision"]:.4f}')
print(f'Best F1           : {metrics["best_f1"]:.4f}  @ threshold={metrics["best_threshold"]:.6f}')

# ROC curve
fpr, tpr, _ = evaluator.roc_curve_data()
prec, rec, _ = evaluator.pr_curve_data()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(fpr, tpr, color='#3498db', lw=2, label=f'AUC = {metrics["roc_auc"]:.4f}')
ax1.plot([0,1],[0,1],'--', color='gray')
ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve — Classical Baseline'); ax1.legend()

ax2.plot(rec, prec, color='#e74c3c', lw=2, label=f'AP = {metrics["average_precision"]:.4f}')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve'); ax2.legend()

plt.tight_layout(); plt.show()

In [ ]:
# ── 2.5  Anomaly map visualisation ────────────────────────────────────────────
anomaly_indices = np.where(labels == 1)[0]
n_show = min(3, len(anomaly_indices))

all_paths = []
for cls_dir in sorted(test_dir.iterdir()):
    if not cls_dir.is_dir(): continue
    all_paths.extend(sorted(cls_dir.glob('*.png')) + sorted(cls_dir.glob('*.jpg')))

fig, axes = plt.subplots(n_show, 3, figsize=(13, n_show * 3.5))
if n_show == 1: axes = [axes]

for row, idx in enumerate(anomaly_indices[:n_show]):
    img  = cv2.imread(str(all_paths[idx]))
    amap = all_amaps[idx]
    amap_norm = (amap - amap.min()) / (amap.max() - amap.min() + 1e-8)
    overlay = AnomalyScorer.overlay(img, amap_norm, alpha=0.5)

    axes[row][0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[row][0].set_title(f'{all_paths[idx].parent.name}/{all_paths[idx].name}', fontsize=7)
    axes[row][0].axis('off')

    axes[row][1].imshow(amap_norm, cmap='hot')
    axes[row][1].set_title(f'Anomaly map (score={scores[idx]:.4f})', fontsize=8)
    axes[row][1].axis('off')

    axes[row][2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    axes[row][2].set_title('Heatmap overlay', fontsize=8)
    axes[row][2].axis('off')

plt.suptitle('Classical Baseline: Anomaly Maps on Defect Images', fontsize=11)
plt.tight_layout(); plt.show()

print(f'\nBaseline ROC-AUC: {metrics["roc_auc"]:.4f}')
print('Target for PatchCore: > 0.95')

## Summary

| Metric | Classical Baseline |
|---|---|
| ROC-AUC | *(see output)* |
| Average Precision | *(see output)* |
| Best F1 | *(see output)* |

This is the **reference** against which PatchCore will be compared in Notebook 06.

## Next step

Open `03_feature_extraction.ipynb` — extract deep CNN features from the training images.
